#### Imports

In [1]:
# Imports
import os
import sys
import shutil
from tqdm import tqdm
from ultralytics import YOLO

#### Path Configurations

In [9]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Path Configuration
samples_to_collect = 2000
original_dataset_name = "soccernet_tracking_collective"
new_dataset_name = f"{samples_to_collect}_soccernet_tracking"
data_directory = os.path.join(project_root, "data")
model_directory = os.path.join(project_root, "models", "detection")
original_dataset_images_path =  os.path.join(data_directory, "detection", original_dataset_name, "images")
original_dataset_labels_path =  os.path.join(data_directory, "detection", original_dataset_name, "labels")

# Smaller images and labels directory setup
small_dataset_path =  os.path.join(data_directory, "detection", new_dataset_name)
small_images_data_directory = os.path.join(small_dataset_path, "images")
small_labels_data_directory = os.path.join(small_dataset_path, "labels")
os.makedirs(small_images_data_directory, exist_ok=True)
os.makedirs(small_labels_data_directory, exist_ok=True)

#### Processor, Device, and Model Setup

In [3]:
# Model Name and Weights
base_model_name = "21-04-2026_07-40_yolo26l"
full_model_weights_path = os.path.join(model_directory , base_model_name, "weights", "best.pt")
detection_model = YOLO(full_model_weights_path) # Can additionally load from a saved point
detection_model.eval()

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_

#### Determing # of files to go through

In [4]:
# Pre-collect files so tqdm knows the total count
jpg_files = [
    (root, file)
    for root, dirs, files in os.walk(original_dataset_images_path)
        for file in files
            if file.endswith(".jpg")
]

#### Uncertainty Configuration

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MODEL_CONFIDENCE_UPPER = 0.80   

#### Utils Import

In [6]:
from utils.uncertainty import gather_least_certain_avg, gather_least_certain_ball

#### Creating a Dataframe for the least certain ball instances

In [ ]:
# uncertainty_df, top_df= gather_least_certain_ball(jpg_files, original_dataset_images_path, detection_model, UNC_LOWER, UNC_UPPER, grab_label=True)
uncertainty_df, top_df= gather_least_certain_avg(jpg_files, original_dataset_images_path, detection_model, MODEL_CONFIDENCE_UPPER, samples_to_collect, grab_label=True)

Scoring images: 100%|██████████| 79500/79500 [45:03<00:00, 29.41img/s]


Top 5000 images selected for annotation.


#### Copying Images over for Annotation

In [ ]:
# ── Copy selected images to annotation directory ───────────────────────────
for _, row in tqdm(top_df.iterrows(), total=len(top_df), desc="Copying images and labels", unit="img"):
    image = row["image"]
    label = row["label"]
    image_dst = os.path.join(small_images_data_directory, os.path.basename(image))
    label_dst = os.path.join(small_labels_data_directory, os.path.basename(label))
    shutil.copy2(image, image_dst)
    shutil.copy2(label, label_dst)

full_csv_path     = os.path.join(small_dataset_path, f"{base_model_name}_uncertainty_scores_all.csv")
selected_csv_path = os.path.join(small_dataset_path, f"{base_model_name}_uncertainty_scores_selected.csv")
top_df.to_csv(selected_csv_path, index=False)

print(f"\n✓ Done.")
print(f"  Copied for annotation:  {len(top_df)}")
print(f"  Full scores   → {full_csv_path}")
print(f"  Selected only → {selected_csv_path}")

Copying images and labels: 100%|██████████| 2000/2000 [00:00<00:00, 4570.68img/s]


✓ Done.
  Copied for annotation:  2000
  Full scores   → /home/tom/Desktop/Programming/Personal/live-footie-formations/data/detection/2000_soccernet_tracking/uncertainty_scores_all.csv
  Selected only → /home/tom/Desktop/Programming/Personal/live-footie-formations/data/detection/2000_soccernet_tracking/uncertainty_scores_selected.csv
